# Panel Data Preparation for R

This notebook prepares the county-year panel datasets used in the subsequent fixed-effects regression analysis in R.

The workflow includes:
- selection of variables required for the panel models;
- validation of the county-year panel structure;
- missing-value and distribution diagnostics;
- standardization of continuous predictors;
- construction of additional labour and mechanization indicators used in alternative model specifications;
- export of the final datasets for regression analysis in R.

In [1]:
import numpy as np
import pandas as pd

### 1. Load Base Panel Dataset

In [2]:
df = pd.read_csv("../../data/processed/analysis_panel_base.csv")
df.head(2)

,an,judet,emig_masc_nr,emig_fem_nr,emigranti_total_nr,emig_15_64_nr,imig_masc_nr,imig_fem_nr,imigranti_total_nr,imigranti_15_64_nr,...,migratie_neta_15_64_nr,rata_migratie_neta_15_64,ocupati_aff_eurostat_mii,ocupati_total_eurostat_mii,ocupati_aff_eurostat_nr,ocupati_total_eurostat_nr,pondere_ocupati_aff_eurostat,schimbare_pondere_ocupati_aff_eurostat,schimbare_ocupati_aff_eurostat_pct,ocupati_aff_eurostat_thousand
0,2012,Alba,143,144,287,222,43,33,76,66,...,-156,-0.679031,28.85,130.90,28850.0,130900.0,22.039725,NaN,NaN,28.85
1,2013,Alba,104,158,262,224,53,41,94,83,...,-141,-0.618020,30.38,131.94,30380.0,131940.0,23.025618,0.985893,5.303293,30.38


In [4]:
df.columns

Index(['an', 'judet', 'emig_masc_nr', 'emig_fem_nr', 'emigranti_total_nr',
       'emig_15_64_nr', 'imig_masc_nr', 'imig_fem_nr', 'imigranti_total_nr',
       'imigranti_15_64_nr', 'populatie_15_64', 'populatie_rurala',
       'populatie_totala', 'pondere_rurala', 'rata_emig_def', 'rata_imig_def',
       'rata_emig_15_64', 'rata_imig_15_64', 'intensitate_migratie',
       'sup_grau_ha', 'sup_orz_orzoaica_ha', 'sup_porumb_boabe_ha',
       'sup_floarea_soarelui_ha', 'sup_rapita_ha', 'sup_soia_boabe_ha',
       'sup_totala_cultivata_ha', 'pondere_grau', 'pondere_orz_orzoaica',
       'pondere_porumb_boabe', 'pondere_floarea_soarelui', 'pondere_rapita',
       'pondere_soia_boabe', 'prod_grau_tone', 'prod_orz_orzoaica_tone',
       'prod_porumb_boabe_tone', 'prod_floarea_soarelui_tone',
       'prod_rapita_tone', 'prod_soia_boabe_tone', 'yield_grau',
       'yield_orz_orzoaica', 'yield_porumb_boabe', 'yield_floarea_soarelui',
       'yield_rapita', 'yield_soia_boabe', 'ocupati_total_nr',


### 2. Variable Selection and Panel Construction

In [5]:
id_cols = [
    "judet",
    "an"
]

yield_main = [
    "yield_grau",
    "yield_porumb_boabe",
    "yield_floarea_soarelui"
]

yield_secondary = [
    "yield_orz_orzoaica",
    "yield_rapita",
    "yield_soia_boabe"
]

migration_vars = [
    "rata_emig_15_64",
    "rata_migratie_neta_15_64",
    "intensitate_migratie"
]

labour_vars = [
    "pondere_ocupati_aff_eurostat",
    "schimbare_pondere_ocupati_aff_eurostat"
]

mechanization_vars = [
    "indice_mecanizare_rezidual",
    "indice_mecanizare_densitate"
]

climate_vars = [
    "temperatura_medie_anuala_C",
    "precipitatii_anuale_mm",
    "schimbare_temp_anuala_C",
    "schimbare_precipitatii_anuala_mm"
]

structure_vars = [
    "pondere_rurala",
    "pondere_grau",
    "pondere_porumb_boabe",
    "pondere_floarea_soarelui",
    "pondere_orz_orzoaica",
    "pondere_rapita",
    "pondere_soia_boabe"
]

final_cols = (
    id_cols
    + yield_main
    + yield_secondary
    + migration_vars
    + labour_vars
    + mechanization_vars
    + climate_vars
    + structure_vars
)

final_cols

['judet',
 'an',
 'yield_grau',
 'yield_porumb_boabe',
 'yield_floarea_soarelui',
 'yield_orz_orzoaica',
 'yield_rapita',
 'yield_soia_boabe',
 'rata_emig_15_64',
 'rata_migratie_neta_15_64',
 'intensitate_migratie',
 'pondere_ocupati_aff_eurostat',
 'schimbare_pondere_ocupati_aff_eurostat',
 'indice_mecanizare_rezidual',
 'indice_mecanizare_densitate',
 'temperatura_medie_anuala_C',
 'precipitatii_anuale_mm',
 'schimbare_temp_anuala_C',
 'schimbare_precipitatii_anuala_mm',
 'pondere_rurala',
 'pondere_grau',
 'pondere_porumb_boabe',
 'pondere_floarea_soarelui',
 'pondere_orz_orzoaica',
 'pondere_rapita',
 'pondere_soia_boabe']

In [7]:
missing_cols = [col for col in final_cols if col not in df.columns]

if missing_cols:
    print("Missing columns:")
    for col in missing_cols:
        print("-", col)
else:
    print("All columns exist")

All columns exist


#### Build the Analysis Panel

In [10]:
panel_brut = df[final_cols].copy()

panel_brut["an"] = panel_brut["an"].astype(int)
panel_brut["judet"] = panel_brut["judet"].astype(str).str.strip()

panel_brut = (
    panel_brut
    .sort_values(["judet", "an"])
    .reset_index(drop=True)
)

print("Panel dimensions:", panel_brut.shape)
print("Number of counties:", panel_brut["judet"].nunique())
print("Period:", panel_brut["an"].min(), "-", panel_brut["an"].max())

display(panel_brut.head())

Panel dimensions: (451, 26)
Number of counties: 41
Period: 2012 - 2022


,judet,an,yield_grau,yield_porumb_boabe,yield_floarea_soarelui,yield_orz_orzoaica,yield_rapita,yield_soia_boabe,rata_emig_15_64,rata_migratie_neta_15_64,...,precipitatii_anuale_mm,schimbare_temp_anuala_C,schimbare_precipitatii_anuala_mm,pondere_rurala,pondere_grau,pondere_porumb_boabe,pondere_floarea_soarelui,pondere_orz_orzoaica,pondere_rapita,pondere_soia_boabe
0,Alba,2012,2.554843,1.977959,1.494949,1.522591,1.002203,0.618421,0.966314,-0.679031,...,763.673799,0.633381,64.982254,41.891503,21.418654,66.176310,2.625462,8.597102,0.708235,0.474237
1,Alba,2013,3.568461,4.124829,0.638382,2.697577,2.219002,1.304965,0.981819,-0.618020,...,960.123806,-0.069853,196.450007,41.847349,26.085441,58.925968,5.210116,7.586994,1.968055,0.223427
2,Alba,2014,4.287926,6.114668,2.514221,2.820071,2.241176,2.868421,0.623067,-0.256297,...,875.316853,0.770750,-84.806953,41.771678,24.540109,60.343788,3.976176,7.693004,3.332328,0.114596
3,Alba,2015,3.999462,4.862960,2.160092,2.841426,2.309748,1.780564,0.781952,-0.388742,...,889.736683,-0.281593,14.419830,41.694146,23.837151,61.371898,5.565138,6.957848,1.813231,0.454733
4,Alba,2016,4.150194,5.935286,2.451675,3.167836,2.996877,2.933735,1.214885,-0.738904,...,1121.871581,-0.631840,232.134898,41.722761,24.673827,59.996405,5.409009,7.255951,1.918842,0.745967


#### Panel Validation

In [25]:
# Check for duplicate county-year observations

duplicates = panel_brut[
    panel_brut.duplicated(subset=["judet", "an"], keep=False)
]

if len(duplicates) > 0:
    print("Duplicate county-year observations found:")
    display(duplicates.sort_values(["judet", "an"]))
else:
    print("No duplicate county-year observations found")

No duplicate county-year observations found


#### Missing-Value Assessment

In [12]:

missing_summary = panel_brut.isna().sum().reset_index()
missing_summary.columns = ["variable", "no_missing"]
missing_summary["pct_missing"] = (missing_summary["no_missing"] / len(panel_brut) * 100).round(2)

missing_summary = missing_summary.sort_values("no_missing", ascending=False)

display(missing_summary)

,variable,no_missing,pct_missing
7,yield_soia_boabe,50,11.09
12,schimbare_pondere_ocupati_aff_eurostat,41,9.09
6,yield_rapita,27,5.99
4,yield_floarea_soarelui,14,3.10
1,an,0,0.00
0,judet,0,0.00
5,yield_orz_orzoaica,0,0.00
3,yield_porumb_boabe,0,0.00
8,rata_emig_15_64,0,0.00
2,yield_grau,0,0.00


### 3. Distribution and Outlier Diagnostics

In [13]:

#evaluating how many complete observations remain for each yield 
model_vars_main = [
    "rata_emig_15_64",
    "pondere_ocupati_aff_eurostat",
    "indice_mecanizare_rezidual",
    "indice_mecanizare_densitate",
    "temperatura_medie_anuala_C",
    "precipitatii_anuale_mm",
    "pondere_rurala"
]

yield_vars = [
    "yield_grau",
    "yield_porumb_boabe",
    "yield_floarea_soarelui",
    "yield_orz_orzoaica",
    "yield_rapita",
    "yield_soia_boabe"
]

complete_cases = []

for y in yield_vars:
    vars_model = [y] + model_vars_main
    n_total = len(panel_brut)
    n_complete = panel_brut[vars_model].dropna().shape[0]
    n_missing = n_total - n_complete
    pct_complete = round(n_complete / n_total * 100, 2)
    
    complete_cases.append({
        "model_yield": y,
        "n_total": n_total,
        "n_complete": n_complete,
        "n_missing": n_missing,
        "pct_complete": pct_complete
    })

complete_cases = pd.DataFrame(complete_cases)

display(complete_cases)

,model_yield,n_total,n_complete,n_missing,pct_complete
0,yield_grau,451,451,0,100.00
1,yield_porumb_boabe,451,451,0,100.00
2,yield_floarea_soarelui,451,437,14,96.90
3,yield_orz_orzoaica,451,451,0,100.00
4,yield_rapita,451,424,27,94.01
5,yield_soia_boabe,451,401,50,88.91


In [14]:
#distribution and extreme-value diagnostics

vars_for_diagnosis = [
    "yield_grau",
    "yield_porumb_boabe",
    "yield_floarea_soarelui",
    "yield_orz_orzoaica",
    "yield_rapita",
    "yield_soia_boabe",
    "rata_emig_15_64",
    "rata_migratie_neta_15_64",
    "intensitate_migratie",
    "pondere_ocupati_aff_eurostat",
    "indice_mecanizare_rezidual",
    "indice_mecanizare_densitate",
    "temperatura_medie_anuala_C",
    "precipitatii_anuale_mm",
    "pondere_rurala"
]

diagnostic_desc = panel_brut[vars_for_diagnosis].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

diagnostic_desc = diagnostic_desc.round(3)

display(diagnostic_desc)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
yield_grau,451.0,3.775,0.884,0.983,1.477,2.288,3.195,3.778,4.402,5.185,5.601,5.995
yield_porumb_boabe,451.0,4.631,1.720,0.252,1.209,1.765,3.466,4.476,5.787,7.582,8.900,10.421
yield_floarea_soarelui,437.0,2.133,0.633,0.427,0.780,1.103,1.697,2.138,2.533,3.149,3.705,3.916
yield_orz_orzoaica,451.0,3.241,0.943,1.017,1.344,1.960,2.606,3.094,3.857,4.819,5.722,7.219
yield_rapita,424.0,2.341,0.605,0.583,0.861,1.218,1.951,2.404,2.753,3.206,3.487,4.030
yield_soia_boabe,401.0,1.969,0.711,0.000,0.561,0.867,1.461,1.917,2.426,3.242,3.612,4.613
rata_emig_15_64,451.0,1.376,0.961,0.153,0.313,0.409,0.767,1.109,1.748,3.031,4.919,8.307
rata_migratie_neta_15_64,451.0,0.764,5.218,-3.941,-2.697,-1.688,-0.961,-0.532,-0.200,10.796,26.071,43.394
intensitate_migratie,451.0,2.645,4.283,0.284,0.386,0.500,0.898,1.366,2.231,11.502,20.381,35.844
pondere_ocupati_aff_eurostat,451.0,28.612,15.815,2.730,4.026,7.494,14.528,26.849,41.542,56.451,62.244,66.145


In [15]:
skew_kurt = pd.DataFrame({
    "variable": vars_for_diagnosis,
    "skewness": [panel_brut[col].skew(skipna=True) for col in vars_for_diagnosis],
    "kurtosis": [panel_brut[col].kurtosis(skipna=True) for col in vars_for_diagnosis]
})

skew_kurt["skewness_abs"] = skew_kurt["skewness"].abs()

skew_kurt = skew_kurt.sort_values("skewness_abs", ascending=False).round(3)

display(skew_kurt)

,variable,skewness,kurtosis,skewness_abs
7,rata_migratie_neta_15_64,4.753,27.082,4.753
8,intensitate_migratie,4.355,22.919,4.355
6,rata_emig_15_64,2.623,11.917,2.623
11,indice_mecanizare_densitate,1.597,2.441,1.597
3,yield_orz_orzoaica,0.613,0.834,0.613
14,pondere_rurala,-0.577,-0.187,0.577
4,yield_rapita,-0.439,-0.015,0.439
9,pondere_ocupati_aff_eurostat,0.343,-0.980,0.343
5,yield_soia_boabe,0.312,0.316,0.312
1,yield_porumb_boabe,0.269,-0.068,0.269


#### Distribution Diagnostics

The strongest distributional asymmetries were identified for:

- `rata_migratie_neta_15_64`: skewness = 4.753
- `intensitate_migratie`: skewness = 4.355
- `rata_emig_15_64`: skewness = 2.623
- `indice_mecanizare_densitate`: skewness = 1.597

These diagnostics were used to identify potentially influential extreme values and to inform subsequent robustness checks.

### 4. Variable Standardization

In [17]:
panel_standardizat = panel_brut.copy()

vars_to_standardize = [
    "rata_emig_15_64",
    "rata_migratie_neta_15_64",
    "intensitate_migratie",
    "pondere_ocupati_aff_eurostat",
    "indice_mecanizare_rezidual",
    "indice_mecanizare_densitate",
    "temperatura_medie_anuala_C",
    "precipitatii_anuale_mm",
    "pondere_rurala"
]

for col in vars_to_standardize:
    mean_col = panel_standardizat[col].mean(skipna=True)
    std_col = panel_standardizat[col].std(skipna=True, ddof=1)
    panel_standardizat[f"z_{col}"] = (panel_standardizat[col] - mean_col) / std_col

print("Dimensiune panel_brut:", panel_brut.shape)
print("Dimensiune panel_standardizat:", panel_standardizat.shape)

z_cols = [f"z_{col}" for col in vars_to_standardize]
display(panel_standardizat[["judet", "an"] + z_cols].head())

Dimensiune panel_brut: (451, 26)
Dimensiune panel_standardizat: (451, 35)


,judet,an,z_rata_emig_15_64,z_rata_migratie_neta_15_64,z_intensitate_migratie,z_pondere_ocupati_aff_eurostat,z_indice_mecanizare_rezidual,z_indice_mecanizare_densitate,z_temperatura_medie_anuala_C,z_precipitatii_anuale_mm,z_pondere_rurala
0,Alba,2012,-0.426704,-0.276617,-0.368774,-0.415592,0.359026,0.349739,-0.815236,-0.003416,-0.954930
1,Alba,2013,-0.410568,-0.264924,-0.372222,-0.353254,0.453970,0.446063,-0.852758,1.088230,-0.958960
2,Alba,2014,-0.783919,-0.195600,-0.440912,-0.501718,0.400563,0.364402,-0.438744,0.616969,-0.965867
3,Alba,2015,-0.618568,-0.220983,-0.409136,-0.676705,0.340269,0.309755,-0.590004,0.697098,-0.972944
4,Alba,2016,-0.168018,-0.288091,-0.331707,-0.793349,0.325312,0.404327,-0.929401,1.987040,-0.970332


In [24]:
#verify standardization

z_cols = [
    "z_rata_emig_15_64",
    "z_rata_migratie_neta_15_64",
    "z_intensitate_migratie",
    "z_pondere_ocupati_aff_eurostat",
    "z_indice_mecanizare_rezidual",
    "z_indice_mecanizare_densitate",
    "z_temperatura_medie_anuala_C",
    "z_precipitatii_anuale_mm",
    "z_pondere_rurala"
]

standardizare_check = panel_standardizat[z_cols].agg(["mean", "std"]).T
standardizare_check = standardizare_check.round(6)

display(standardizare_check)

,mean,std
z_rata_emig_15_64,-0.0,1.0
z_rata_migratie_neta_15_64,0.0,1.0
z_intensitate_migratie,0.0,1.0
z_pondere_ocupati_aff_eurostat,0.0,1.0
z_indice_mecanizare_rezidual,-0.0,1.0
z_indice_mecanizare_densitate,0.0,1.0
z_temperatura_medie_anuala_C,0.0,1.0
z_precipitatii_anuale_mm,0.0,1.0
z_pondere_rurala,0.0,1.0


### 5. Final Panel Preparation for R

Additional agricultural labour indicators were constructed from Eurostat AFF employment data to support alternative specifications of H2.


- aff_workers_per_1000ha

- ha_per_aff_worker

- z_aff_workers_per_1000ha

- z_ha_per_aff_worker

In [26]:
import pandas as pd
import numpy as np



df_panel = df.copy()

df_panel["judet"] = df_panel["judet"].astype(str).str.strip()
df_panel = df_panel[df_panel["judet"].str.lower() != "bucuresti"].copy()
df_panel["an"] = pd.to_numeric(df_panel["an"], errors="coerce").astype("Int64")
df_panel = df_panel.sort_values(["judet", "an"]).reset_index(drop=True)



cols_final_raw = [
    "judet",
    "an",

    # productivity
    "yield_grau",
    "yield_porumb_boabe",
    "yield_floarea_soarelui",

    # migration
    "rata_emig_15_64",
    "rata_imig_15_64",
    "rata_migratie_neta_15_64",

    # AFF employment - Eurostat
    "ocupati_aff_eurostat_nr",
    "ocupati_total_eurostat_nr",
    "pondere_ocupati_aff_eurostat",
    "schimbare_pondere_ocupati_aff_eurostat",
    "schimbare_ocupati_aff_eurostat_pct",

    # mechanization
    "indice_mecanizare_rezidual",
    "indice_mecanizare_densitate",

    # climate
    "temperatura_medie_anuala_C",
    "precipitatii_anuale_mm",

    # rural / agricultural structure
    "pondere_rurala",
    "sup_totala_cultivata_ha",
    "pondere_grau",
    "pondere_porumb_boabe",
    "pondere_floarea_soarelui"
]

missing_cols = [col for col in cols_final_raw if col not in df_panel.columns]

if missing_cols:
    raise ValueError(f"The following columns are missing from df: {missing_cols}")

df_final = df_panel[cols_final_raw].copy()




numeric_cols = [col for col in df_final.columns if col not in ["judet", "an"]]

for col in numeric_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors="coerce")




In [27]:

# Log transformation of selected cultivated area
df_final["log_sup_totala_cultivata"] = np.where(
    df_final["sup_totala_cultivata_ha"] > 0,
    np.log(df_final["sup_totala_cultivata_ha"]),
    np.nan
)

# AFF workers per 1,000 ha
# Higher values = more AFF workers relative to cultivated area
df_final["aff_workers_per_1000ha"] = np.where(
    df_final["sup_totala_cultivata_ha"] > 0,
    df_final["ocupati_aff_eurostat_nr"] / df_final["sup_totala_cultivata_ha"] * 1000,
    np.nan
)

# Hectares per AFF worker
# Higher values = more cultivated area per AFF worker
df_final["ha_per_aff_worker"] = np.where(
    df_final["ocupati_aff_eurostat_nr"] > 0,
    df_final["sup_totala_cultivata_ha"] / df_final["ocupati_aff_eurostat_nr"],
    np.nan
)

# Annual change in AFF employment
# Calculated from Eurostat AFF employment counts
df_final["change_aff_employment_number"] = (
    df_final
    .groupby("judet")["ocupati_aff_eurostat_nr"]
    .diff()
)

# AFF employment decline indicators
# Higher values = larger decline in AFF employment
df_final["decline_aff_employment_number"] = -df_final["change_aff_employment_number"]
df_final["decline_aff_employment_pct"] = -df_final["schimbare_ocupati_aff_eurostat_pct"]
df_final["decline_aff_employment_share"] = -df_final["schimbare_pondere_ocupati_aff_eurostat"]



In [28]:

# Standardization


def zscore(series):
    sd = series.std(ddof=1)
    if sd == 0 or pd.isna(sd):
        return np.nan
    return (series - series.mean()) / sd


vars_to_standardize = [
    # main variables
    "rata_emig_15_64",
    "pondere_ocupati_aff_eurostat",
    "schimbare_pondere_ocupati_aff_eurostat",
    "indice_mecanizare_rezidual",
    "temperatura_medie_anuala_C",
    "precipitatii_anuale_mm",
    "pondere_rurala",
    "log_sup_totala_cultivata",
    "pondere_grau",
    "pondere_porumb_boabe",
    "pondere_floarea_soarelui",

    # H2 variables - constructed only from Eurostat AFF data
    "ocupati_aff_eurostat_nr",
    "aff_workers_per_1000ha",
    "ha_per_aff_worker",
    "change_aff_employment_number",
    "schimbare_ocupati_aff_eurostat_pct",
    "schimbare_pondere_ocupati_aff_eurostat",
    "decline_aff_employment_number",
    "decline_aff_employment_pct",
    "decline_aff_employment_share",

    # robustness variables
    "indice_mecanizare_densitate",
    "rata_migratie_neta_15_64",
    "rata_imig_15_64"
]

for col in vars_to_standardize:
    df_final[f"z_{col}"] = zscore(df_final[col])



### 6. Final Dataset Preparation for R

In [29]:



cols_final_R = [
    "judet",
    "an",

    # unstandardized outcome variables
    "yield_grau",
    "yield_porumb_boabe",
    "yield_floarea_soarelui",

    # raw variables useful for descriptive analysis
    "rata_emig_15_64",
    "rata_imig_15_64",
    "rata_migratie_neta_15_64",

    # AFF employment - Eurostat
    "ocupati_aff_eurostat_nr",
    "ocupati_total_eurostat_nr",
    "pondere_ocupati_aff_eurostat",
    "schimbare_pondere_ocupati_aff_eurostat",
    "schimbare_ocupati_aff_eurostat_pct",

    # H2 variables constructed from AFF data
    "aff_workers_per_1000ha",
    "ha_per_aff_worker",
    "change_aff_employment_number",
    "decline_aff_employment_number",
    "decline_aff_employment_pct",
    "decline_aff_employment_share",

    # mechanization
    "indice_mecanizare_rezidual",
    "indice_mecanizare_densitate",

    # climate
    "temperatura_medie_anuala_C",
    "precipitatii_anuale_mm",

    # rural / agricultural structure
    "pondere_rurala",
    "sup_totala_cultivata_ha",
    "log_sup_totala_cultivata",
    "pondere_grau",
    "pondere_porumb_boabe",
    "pondere_floarea_soarelui",

    # standardized main variables
    "z_rata_emig_15_64",
    "z_pondere_ocupati_aff_eurostat",
    "z_schimbare_pondere_ocupati_aff_eurostat",
    "z_indice_mecanizare_rezidual",
    "z_temperatura_medie_anuala_C",
    "z_precipitatii_anuale_mm",
    "z_pondere_rurala",
    "z_log_sup_totala_cultivata",
    "z_pondere_grau",
    "z_pondere_porumb_boabe",
    "z_pondere_floarea_soarelui",

    # standardized H2 variables - constructed only from AFF data
    "z_ocupati_aff_eurostat_nr",
    "z_aff_workers_per_1000ha",
    "z_ha_per_aff_worker",
    "z_change_aff_employment_number",
    "z_schimbare_ocupati_aff_eurostat_pct",
    "z_decline_aff_employment_number",
    "z_decline_aff_employment_pct",
    "z_decline_aff_employment_share",

    # standardized robustness variables
    "z_indice_mecanizare_densitate",
    "z_rata_migratie_neta_15_64",
    "z_rata_imig_15_64"
]

df_R = df_final[cols_final_R].copy()


In [30]:

# Rename columns to English



rename_dict = {
    # identifiers
    "judet": "county",
    "an": "year",

    # productivity
    "yield_grau": "wheat_yield",
    "yield_porumb_boabe": "maize_yield",
    "yield_floarea_soarelui": "sunflower_yield",

    # migration
    "rata_emig_15_64": "emigration_rate_15_64",
    "rata_imig_15_64": "immigration_rate_15_64",
    "rata_migratie_neta_15_64": "net_migration_rate_15_64",

    # AFF employment - Eurostat
    "ocupati_aff_eurostat_nr": "aff_employment_number",
    "ocupati_total_eurostat_nr": "total_employment_number",
    "pondere_ocupati_aff_eurostat": "aff_employment_share",
    "schimbare_pondere_ocupati_aff_eurostat": "change_aff_employment_share",
    "schimbare_ocupati_aff_eurostat_pct": "change_aff_employment_pct",

    # H2 variables constructed from AFF data
    "aff_workers_per_1000ha": "aff_workers_per_1000ha",
    "ha_per_aff_worker": "ha_per_aff_worker",
    "change_aff_employment_number": "change_aff_employment_number",
    "decline_aff_employment_number": "decline_aff_employment_number",
    "decline_aff_employment_pct": "decline_aff_employment_pct",
    "decline_aff_employment_share": "decline_aff_employment_share",

    # mechanization
    "indice_mecanizare_rezidual": "residual_mechanization_index",
    "indice_mecanizare_densitate": "density_mechanization_index",

    # climate
    "temperatura_medie_anuala_C": "annual_mean_temperature_c",
    "precipitatii_anuale_mm": "annual_precipitation_mm",

    # rural / agricultural structure
    "pondere_rurala": "rural_population_share",
    "sup_totala_cultivata_ha": "selected_crop_area_ha",
    "log_sup_totala_cultivata": "log_selected_crop_area",

    # crop area shares
    "pondere_grau": "wheat_area_share",
    "pondere_porumb_boabe": "maize_area_share",
    "pondere_floarea_soarelui": "sunflower_area_share",

    # standardized main variables
    "z_rata_emig_15_64": "z_emigration_rate_15_64",
    "z_pondere_ocupati_aff_eurostat": "z_aff_employment_share",
    "z_schimbare_pondere_ocupati_aff_eurostat": "z_change_aff_employment_share",
    "z_indice_mecanizare_rezidual": "z_residual_mechanization_index",
    "z_temperatura_medie_anuala_C": "z_annual_mean_temperature_c",
    "z_precipitatii_anuale_mm": "z_annual_precipitation_mm",
    "z_pondere_rurala": "z_rural_population_share",
    "z_log_sup_totala_cultivata": "z_log_selected_crop_area",

    # standardized crop area shares
    "z_pondere_grau": "z_wheat_area_share",
    "z_pondere_porumb_boabe": "z_maize_area_share",
    "z_pondere_floarea_soarelui": "z_sunflower_area_share",

    # standardized H2 variables - constructed only from AFF data
    "z_ocupati_aff_eurostat_nr": "z_aff_employment_number",
    "z_aff_workers_per_1000ha": "z_aff_workers_per_1000ha",
    "z_ha_per_aff_worker": "z_ha_per_aff_worker",
    "z_change_aff_employment_number": "z_change_aff_employment_number",
    "z_schimbare_ocupati_aff_eurostat_pct": "z_change_aff_employment_pct",
    "z_decline_aff_employment_number": "z_decline_aff_employment_number",
    "z_decline_aff_employment_pct": "z_decline_aff_employment_pct",
    "z_decline_aff_employment_share": "z_decline_aff_employment_share",

    # standardized robustness variables
    "z_indice_mecanizare_densitate": "z_density_mechanization_index",
    "z_rata_migratie_neta_15_64": "z_net_migration_rate_15_64",
    "z_rata_imig_15_64": "z_immigration_rate_15_64"
}

df_R = df_R.rename(columns=rename_dict)



In [31]:

# Validation checks


print("Final dataset dimensions:")
print(df_R.shape)

print("\nAvailable years:")
print(sorted(df_R["year"].dropna().unique()))

print("\nNumber of counties:")
print(df_R["county"].nunique())

print("\nObservations per county:")
print(df_R.groupby("county")["year"].nunique().describe())

missing_table = (
    df_R.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "variable", 0: "missing_n"})
)

missing_table["missing_pct"] = missing_table["missing_n"] / len(df_R) * 100

print("\nMissing values:")
print(missing_table.sort_values("missing_pct", ascending=False).to_string(index=False))

print("\nFinal columns:")
print(df_R.columns.tolist())

# Quick validation of the new H2 variables constructed only from AFF data
h2_new_vars = [
    "aff_employment_number",
    "aff_workers_per_1000ha",
    "ha_per_aff_worker",
    "change_aff_employment_number",
    "change_aff_employment_pct",
    "change_aff_employment_share",
    "decline_aff_employment_number",
    "decline_aff_employment_pct",
    "decline_aff_employment_share",
    "z_aff_employment_number",
    "z_aff_workers_per_1000ha",
    "z_ha_per_aff_worker",
    "z_change_aff_employment_number",
    "z_change_aff_employment_pct",
    "z_change_aff_employment_share",
    "z_decline_aff_employment_number",
    "z_decline_aff_employment_pct",
    "z_decline_aff_employment_share"
]

print("\nValidation of new H2 variables constructed only from AFF data:")
print(df_R[h2_new_vars].describe().T)



Final dataset dimensions:
(451, 51)

Available years:
[np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

Number of counties:
41

Observations per county:
count    41.0
mean     11.0
std       0.0
min      11.0
25%      11.0
50%      11.0
75%      11.0
max      11.0
Name: year, dtype: float64

Missing values:
                       variable  missing_n  missing_pct
   decline_aff_employment_share         41     9.090909
  decline_aff_employment_number         41     9.090909
   change_aff_employment_number         41     9.090909
      change_aff_employment_pct         41     9.090909
    change_aff_employment_share         41     9.090909
     decline_aff_employment_pct         41     9.090909
 z_decline_aff_employment_share         41     9.090909
 z_change_aff_employment_number         41     9.090909
z_decline_aff_employment_number         41     9.090909
   

In [32]:

#Export


df_R.to_csv("dataset_final_R_panel_en_H2_AFF.csv", index=False, encoding="utf-8-sig")

print("\nFile exported: dataset_final_R_panel_en_H2_AFF.csv")


File exported: dataset_final_R_panel_en_H2_AFF.csv
